In [1]:
import pandas as pd

In [2]:
raw = pd.read_csv("/Users/jasleenkaur/Event-Driven-Congestion/data/raw/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv")

In [3]:
TRUSTWORTHY_CAUSES = [
    "accident", "tree_fall", "congestion",
]

In [4]:
raw["start_datetime"] = pd.to_datetime(raw["start_datetime"], utc=True, errors="coerce")
raw["closed_datetime"] = pd.to_datetime(raw["closed_datetime"], utc=True, errors="coerce")

In [5]:
df = raw[
    (raw["status"] == "closed")
    & raw["start_datetime"].notna()
    & raw["closed_datetime"].notna()
    & raw["event_cause"].isin(TRUSTWORTHY_CAUSES)
].copy()

In [6]:
df["duration_min"] = (df["closed_datetime"] - df["start_datetime"]).dt.total_seconds() / 60

In [7]:
df = df[(df["duration_min"] >= 0) & (df["duration_min"] <= 7 * 24 * 60)]

In [8]:
print(df.shape)

(249, 47)


In [9]:
print(df.groupby("event_cause")["duration_min"].describe())

             count         mean          std       min        25%         50%  \
event_cause                                                                     
accident      87.0    47.933117    33.584104  0.849123  26.628917   40.035959   
congestion    22.0    74.678392    47.920572  1.561530  41.470969   71.535313   
tree_fall    140.0  1329.391209  2234.651009  1.080537  62.520828  217.805818   

                     75%          max  
event_cause                            
accident       61.554296   216.126667  
congestion    113.889402   157.831133  
tree_fall    1447.446064  9465.174807  


In [10]:
df["duration_threshold"] = df.groupby("event_cause")["duration_min"].transform(lambda s: s.quantile(0.75))
df["high_duration_risk"] = (df["duration_min"] > df["duration_threshold"]).astype(int)

In [11]:
print(df["high_duration_risk"].value_counts(normalize=True))

high_duration_risk
0    0.746988
1    0.253012
Name: proportion, dtype: float64


In [12]:
df["hour"] = df["start_datetime"].dt.hour
df["month"] = df["start_datetime"].dt.month
df["is_weekend"] = (df["start_datetime"].dt.weekday >= 5).astype(int)
df["requires_road_closure"] = df["requires_road_closure"].astype(int)

In [13]:
keep_cols = [
    "id", "event_type", "event_cause", "priority", "requires_road_closure",
    "hour", "month", "is_weekend", "corridor", "zone",
    "duration_min", "high_duration_risk",
]
df[keep_cols].to_csv("../data/processed/duration_dataset.csv", index=False)